# OneVoice V2 — denoiser A/B
Runs the measured passthrough baseline on every compatible Colab runtime. DeepFilterNet is an optional quality-ceiling candidate: if its native dependency cannot install on the current Python version, the notebook reports the install log and continues with passthrough rather than fabricating a denoiser result.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
WORK_ROOT = MYDRIVE / 'OneVoice'
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.environ['HF_HOME'] = str(WORK_ROOT / 'model_cache/huggingface')
os.environ['PYTHONUNBUFFERED'] = '1'
os.chdir(REPO)
DATASET_ROOT = MYDRIVE / 'onevoice_audio_v1'
MANIFEST = DATASET_ROOT / 'manifest.jsonl'
if not MANIFEST.is_file(): raise FileNotFoundError('Run colab_data_audit_v2.ipynb first to create/recover manifest.jsonl')
REPORT_ROOT = WORK_ROOT / 'reports/denoiser'

def run_streaming(command, label):
    print(f'\n[{label}] > ' + ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    print(f'[{label}] exit code: {code}', flush=True)
    if code:
        raise RuntimeError(f'{label} failed; the complete subprocess log is printed above.')

print('Source:', REPO, '| Data:', MANIFEST, '| Reports:', REPORT_ROOT)


## Cài dependency
Passthrough cần ASR core. DeepFilterNet cài riêng vì release cũ có thể chưa tương thích Python runtime hiện tại của Colab.

In [ ]:
def pip_install(packages, *, required):
    command = [sys.executable, '-m', 'pip', 'install', '--prefer-binary', *packages]
    print('>', ' '.join(command), flush=True)
    result = subprocess.run(command)
    if result.returncode and required:
        raise RuntimeError('Required dependency install failed; see pip log above')
    return result.returncode == 0

pip_install(['numpy', 'PyYAML', 'soundfile', 'librosa', 'scipy', 'huggingface_hub'], required=True)
pip_install(['sherpa-onnx'], required=True)
HAS_DEEPFILTER = pip_install(['deepfilternet==0.5.6'], required=False)
if HAS_DEEPFILTER:
    print('> validating DeepFilterNet import', flush=True)
    HAS_DEEPFILTER = subprocess.run([sys.executable, '-c', 'from df.enhance import enhance, init_df']).returncode == 0
CANDIDATES = ('passthrough', 'deepfilter') if HAS_DEEPFILTER else ('passthrough',)
if not HAS_DEEPFILTER:
    print('DeepFilterNet unavailable on this runtime; running passthrough only. This is a FALLBACK result, not a failed ASR benchmark.')
print('Candidates:', CANDIDATES)


In [ ]:
for backend in CANDIDATES:
    for audio in ('clean', 'noisy'):
        report_dir = REPORT_ROOT / backend / audio
        print(f'Running {backend} / {audio} → {report_dir}', flush=True)
        required = ('aggregate.json', 'predictions.csv', 'run_manifest.json')
        if all((report_dir / name).is_file() for name in required):
            print(f'ASR {backend}/{audio} already complete on Drive; skipping.', flush=True)
            continue
        run_streaming([sys.executable, 'scripts/benchmark_asr_v2.py', str(MANIFEST), '--direction', 'vi2en', '--split', 'test', '--audio', audio, '--denoiser', backend, '--report-dir', str(report_dir)], f'ASR {backend}/{audio}')


In [ ]:
for backend in CANDIDATES:
    if backend == 'passthrough': continue
    subprocess.run([sys.executable, 'scripts/evaluate_denoiser_gate.py', '--baseline-clean', str(REPORT_ROOT / 'passthrough/clean/aggregate.json'), '--baseline-noisy', str(REPORT_ROOT / 'passthrough/noisy/aggregate.json'), '--candidate-clean', str(REPORT_ROOT / backend / 'clean/aggregate.json'), '--candidate-noisy', str(REPORT_ROOT / backend / 'noisy/aggregate.json'), '--report', str(REPORT_ROOT / backend / 'gate.json')], check=False)


In [ ]:
import json
{f'{backend}/{audio}': json.loads((REPORT_ROOT / backend / audio / 'aggregate.json').read_text(encoding='utf-8')) for backend in CANDIDATES for audio in ('clean','noisy')}
